# Eksperimen — Telco Customer Churn
**Nama:** Sheren Failla

Prediksi pelanggan yang akan berhenti berlangganan (churn) dari data 7.043 pelanggan telco (IBM sample dataset).
Alur sesuai template MSML: **Data Loading → EDA → Preprocessing**.

## 1. Import Library

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)

## 2. Data Loading

In [2]:
df = pd.read_csv('../telco_raw.csv')
print('Shape:', df.shape)
df.head()

Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


**Temuan awal:** `TotalCharges` bertipe *object* padahal seharusnya numerik — indikasi ada nilai non-angka yang perlu diinvestigasi di EDA.

## 3. Exploratory Data Analysis (EDA)

### 3.1 Statistik deskriptif

In [4]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


### 3.2 Distribusi target (Churn)

In [5]:
churn_rate = (df['Churn'] == 'Yes').mean()
print(f'Churn rate: {churn_rate:.1%}  ->  dataset imbalanced')
sns.countplot(x='Churn', data=df)
plt.title('Distribusi Churn')
plt.show()

Churn rate: 26.5%  ->  dataset imbalanced


### 3.3 Investigasi TotalCharges

In [6]:
blank = df[df['TotalCharges'].str.strip() == '']
print(f'Baris dengan TotalCharges kosong: {len(blank)}')
print('Nilai tenure pada baris tersebut:', blank['tenure'].unique())

Baris dengan TotalCharges kosong: 11
Nilai tenure pada baris tersebut: [0]


Semua baris kosong ternyata pelanggan dengan `tenure = 0` — pelanggan baru yang **belum pernah ditagih**.
Keputusan preprocessing: konversi ke numerik dan isi dengan **0** (bukan median), karena secara bisnis memang belum ada tagihan.

### 3.4 Churn berdasarkan tipe kontrak

In [7]:
ct = pd.crosstab(df['Contract'], df['Churn'], normalize='index')
print(ct)
sns.countplot(x='Contract', hue='Churn', data=df)
plt.title('Churn vs Tipe Kontrak')
plt.show()

Churn                 No       Yes
Contract                          
Month-to-month  0.572903  0.427097
One year        0.887305  0.112695
Two year        0.971681  0.028319


Pelanggan kontrak *month-to-month* churn jauh lebih tinggi (~43%) dibanding kontrak 1–2 tahun (<12%) — fitur `Contract` diprediksi sangat informatif.

### 3.5 Distribusi fitur numerik

In [8]:
num = df[['tenure', 'MonthlyCharges']].copy()
num['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
num.hist(figsize=(12, 4), bins=30, layout=(1, 3))
plt.tight_layout()
plt.show()

### 3.6 Korelasi fitur numerik

In [9]:
num['SeniorCitizen'] = df['SeniorCitizen']
num['Churn'] = (df['Churn'] == 'Yes').astype(int)
plt.figure(figsize=(7, 5))
sns.heatmap(num.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Korelasi Fitur Numerik')
plt.show()

`tenure` berkorelasi **negatif** dengan churn (makin lama berlangganan, makin loyal) — konsisten dengan temuan kontrak.

## 4. Preprocessing
Berdasarkan EDA: (1) drop `customerID`, (2) perbaiki `TotalCharges`, (3) encoding biner & one-hot, (4) split stratified, (5) standardisasi numerik.

### 4.1 Cleaning

In [10]:
df_clean = df.drop(columns=['customerID']).copy()
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce').fillna(0)
print('Missing tersisa:', df_clean.isnull().sum().sum())
print('Duplikat:', df_clean.duplicated().sum())

Missing tersisa: 0
Duplikat: 22


### 4.2 Encoding

In [11]:
binary_map = {'Yes': 1, 'No': 0, 'Female': 1, 'Male': 0}
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
for col in binary_cols:
    df_clean[col] = df_clean[col].map(binary_map)

multi_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
              'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
              'Contract', 'PaymentMethod']
df_encoded = pd.get_dummies(df_clean, columns=multi_cols, drop_first=True, dtype=int)
print('Shape setelah encoding:', df_encoded.shape)
df_encoded.head()

Shape setelah encoding: (7043, 31)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,MultipleLines_No phone service,MultipleLines_Yes,InternetService_Fiber optic,InternetService_No,OnlineSecurity_No internet service,OnlineSecurity_Yes,OnlineBackup_No internet service,OnlineBackup_Yes,DeviceProtection_No internet service,DeviceProtection_Yes,TechSupport_No internet service,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,1,0,1,0,1,0,1,29.85,29.85,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0
1,0,0,0,0,34,1,0,56.95,1889.50,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1
2,0,0,0,0,2,1,1,53.85,108.15,1,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1
3,0,0,0,0,45,0,0,42.30,1840.75,0,1,0,0,0,0,1,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0
4,1,0,0,0,2,1,1,70.70,151.65,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0


### 4.3 Split train-test (stratified)

In [12]:
X = df_encoded.drop(columns=['Churn'])
y = df_encoded['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train:', X_train.shape, '| Test:', X_test.shape)
print(f'Churn rate train: {y_train.mean():.3f} | test: {y_test.mean():.3f}')

Train: (5634, 30) | Test: (1409, 30)
Churn rate train: 0.265 | test: 0.265


### 4.4 Standardisasi fitur numerik kontinu
Scaler di-*fit* hanya pada train untuk menghindari data leakage.

In [13]:
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])
X_train[numeric_cols].describe()

,tenure,MonthlyCharges,TotalCharges
count,5.634000e+03,5.634000e+03,5.634000e+03
mean,-1.008935e-17,-2.402527e-16,2.522338e-17
std,1.000089e+00,1.000089e+00,1.000089e+00
min,-1.322329e+00,-1.544028e+00,-1.008922e+00
25%,-9.559779e-01,-9.711977e-01,-8.321009e-01
50%,-1.418632e-01,1.848336e-01,-3.968446e-01
75%,9.164859e-01,8.319124e-01,6.741944e-01
max,1.608483e+00,1.785939e+00,2.801869e+00


### 4.5 Simpan hasil preprocessing

In [14]:
import os
os.makedirs('telco_preprocessing', exist_ok=True)

train_out = X_train.copy(); train_out['Churn'] = y_train.values
test_out  = X_test.copy();  test_out['Churn']  = y_test.values

train_out.to_csv('telco_preprocessing/telco_train.csv', index=False)
test_out.to_csv('telco_preprocessing/telco_test.csv', index=False)
print('Tersimpan: telco_train.csv & telco_test.csv')

Tersimpan: telco_train.csv & telco_test.csv


## 5. Kesimpulan
- 7.043 pelanggan, 21 kolom; target imbalanced (26,5% churn).
- `TotalCharges` kosong = pelanggan baru (tenure 0) → diisi 0, bukan median.
- Kontrak *month-to-month* dan `tenure` rendah adalah sinyal churn terkuat.
- Encoding menghasilkan 30 fitur siap latih; scaling fit hanya di train (tanpa leakage).
- Seluruh langkah dikonversi ke `automate_Sheren-Failla.py`.